# Balancing Accuracy, Efficiency, and Robustness: An Empirical Study of Quantized CNN Models for Edge Deployment

**Author:** AI Researcher / Machine Learning Engineer  
**Project:** Baseline Empirical Study for Edge AI Deployment  
**Dataset:** MNIST Benchmark  
**Model Architecture:** ResNet-18 (FP32 baseline vs. INT8 Post-Training Quantized)  

---

### Research Abstract & Objectives
Deploying Deep Neural Networks on resource-constrained edge devices (e.g., microcontrollers, IoT gateways, mobile devices) requires balancing **predictive accuracy**, **computational efficiency**, and **adversarial robustness**. 

This notebook establishes the **pilot baseline experiment** for an academic study evaluating the impact of Post-Training Quantization (PTQ) on Convolutional Neural Networks (CNNs).

#### Experimental Evaluation Metrics:
1. **Clean Predictive Accuracy (%)**
2. **Adversarial Accuracy under FGSM Attack (%)**
3. **Model Footprint Size (MB)**
4. **Inference Latency (ms per image)**
5. **Model Compression Ratio**

---

## 1. Environment Setup and Library Imports

### What is being done
We import all core standard libraries required for PyTorch model development, data loading, quantizing models, measuring sizes, latency timing, mathematical arrays, data frames, and graph generation.

### Why it is necessary
Explicitly importing only standard allowed libraries ensures runtime compatibility and avoids unneeded third-party dependencies.

### How it contributes to the research
Establishes the foundational software stack for the baseline research experiment.

In [ ]:
import os
import time
import copy
import tempfile
import random
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from torchvision import transforms
from torchvision import models
from torch.utils.data import DataLoader
import torch.ao.quantization as quantization

print("All required standard libraries imported successfully.")

## 2. Random Seed Configuration for Academic Reproducibility

### What is being done
We set a deterministic random seed (`42`) across Python's built-in `random` module, `NumPy`, and `PyTorch` (CPU and CUDA backends).

### Why it is necessary
In deep learning experiments, stochastic operations such as weight initialization, data shuffling, and dropout can introduce variability between experimental runs. Fixing the random seed ensures that experimental results remain identical across different execution environments.

### How it contributes to the research
Academic rigor requires strict reproducibility. By fixing seed `42`, fellow researchers can independently re-run this notebook and obtain identical quantitative results.

In [ ]:
# Define fixed seed value for full reproducibility
SEED = 42

# Set random seed for Python built-in random module
random.seed(SEED)

# Set random seed for NumPy
np.random.seed(SEED)

# Set random seed for PyTorch CPU operations
torch.manual_seed(SEED)

# Set random seed for PyTorch GPU operations (if CUDA available)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Ensure deterministic algorithmic behavior in CuDNN backend
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed set to {SEED} for Python, NumPy, and PyTorch.")

## 3. Hardware Compute Device Selection

### What is being done
We automatically query the host system for available CUDA-capable GPU hardware. If available, execution targets `cuda`; otherwise, execution falls back to `cpu`.

### Why it is necessary
Deep learning model training and inference benchmarks depend directly on the execution device. Automatic device detection enables seamless portability across machines with or without GPU acceleration.

### How it contributes to the research
Documenting the compute backend ensures clear measurement contexts for inference latency and compute benchmarks.

In [ ]:
# Detect available hardware compute device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using device: cuda")
else:
    device = torch.device("cpu")
    print("Using device: cpu")

## 4. Dataset Setup and Automated Downloading (MNIST)

### What is being done
We create a local directory `./data/` and automatically download the standard MNIST handwritten digit dataset using `torchvision.datasets.MNIST`.

### Why it is necessary
The notebook must run autonomously without requiring pre-existing local datasets. Storing data in `./data/` ensures clean project organization.

### How it contributes to the research
Using a standard benchmark dataset (MNIST: 60,000 training samples, 10,000 testing samples across 10 digit classes) provides a universally recognized baseline for pilot CNN quantization studies.

In [ ]:
# Define local data directory
data_dir = "./data/"

# Ensure the data directory exists
os.makedirs(data_dir, exist_ok=True)

# Download training dataset
raw_train_data = datasets.MNIST(root=data_dir, train=True, download=True)

# Download testing dataset
raw_test_data = datasets.MNIST(root=data_dir, train=False, download=True)

print("MNIST dataset successfully downloaded and verified in ./data/")

## 5. Defining Data Transformation Pipelines

### What is being done
We define preprocessing pipelines for training and testing data using `torchvision.transforms`:
- **Resize:** Resizes 28×28 MNIST images to 224×224 pixels.
- **ToTensor:** Converts PIL images to PyTorch float tensors normalized to `[0.0, 1.0]`.
- **Normalize:** Standardizes pixel values using MNIST channel mean `0.1307` and standard deviation `0.3081`.

### Why it is necessary
ResNet-18 expects 224×224 input spatial dimensions. Channel normalization stabilizes gradient flow and accelerates neural network convergence during training.

### How it contributes to the research
Standardizing image pre-processing guarantees consistent feature representation across both FP32 and INT8 model evaluation phases.

In [ ]:
# Define target image dimensions and normalization statistics
IMAGE_SIZE = (224, 224)
MNIST_MEAN = (0.1307,)
MNIST_STD = (0.3081,)

# Training data transformation pipeline
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD)
])

# Testing data transformation pipeline
test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD)
])

print("Data transformation pipelines defined successfully.")

## 6. PyTorch Dataset Instantiation

### What is being done
We load the MNIST training and testing partitions using `torchvision.datasets.MNIST`, binding the defined transformation pipelines.

### Why it is necessary
Applying transforms at dataset instantiation ensures that mini-batches fetched during training and testing are transformed dynamically on-the-fly.

### How it contributes to the research
Maintains clean separation between raw data storage and preprocessed tensor streams.

In [ ]:
# Create transformed training dataset
train_dataset = datasets.MNIST(
    root=data_dir,
    train=True,
    transform=train_transform,
    download=False
)

# Create transformed testing dataset
test_dataset = datasets.MNIST(
    root=data_dir,
    train=False,
    transform=test_transform,
    download=False
)

print(f"Train dataset created: {len(train_dataset)} samples")
print(f"Test dataset created: {len(test_dataset)} samples")

## 7. DataLoaders Construction

### What is being done
We instantiate PyTorch `DataLoader` objects for the training and testing datasets with a mini-batch size of `64`.
- `train_loader`: `shuffle=True` to randomize mini-batch ordering.
- `test_loader`: `shuffle=False` for deterministic evaluation order.

### Why it is necessary
DataLoaders handle batching, memory buffering, and optional mini-batch shuffling during training and evaluation.

### How it contributes to the research
Batching ensures efficient hardware memory utilization and stable stochastic gradient estimates during optimization.

In [ ]:
# Define mini-batch size
BATCH_SIZE = 64

# Construct DataLoader for training set
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Construct DataLoader for testing set
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Train loader initialized with batch size {BATCH_SIZE}")
print(f"Test loader initialized with batch size {BATCH_SIZE}")

## 8. Dataset Statistics and Metadata Summary

### What is being done
We inspect and display fundamental dataset metadata including sample counts, total class count, and string class names.

### Why it is necessary
Verifying dataset dimensions and class mappings validates dataset integrity before feeding data into the neural network.

### How it contributes to the research
Ensures proper configuration of model output classification dimensions.

In [ ]:
# Determine training and testing sample counts
num_train_samples = len(train_dataset)
num_test_samples = len(test_dataset)

# Extract class names and total number of classes
class_names = [str(i) for i in range(10)]
num_classes = len(class_names)

# Print dataset summary statistics
print("=== DATASET SUMMARY METADATA ===")
print(f"Training samples: {num_train_samples}")
print(f"Testing samples:  {num_test_samples}")
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")

## 9. Data Visualization (Sample Image Inspection)

### What is being done
We extract a sample mini-batch from the dataset, un-normalize the image tensors back to standard visual range, and plot `8` random sample images alongside their corresponding ground-truth digit labels using `matplotlib`.

### Why it is necessary
Visual inspection provides a crucial sanity check that images are resized, normalized, and labeled correctly.

### How it contributes to the research
Confirms visual data quality and alignment before model training.

In [ ]:
# Obtain a batch of training images and labels
images, labels = next(iter(train_loader))

# Create a figure for displaying 8 sample images
plt.figure(figsize=(12, 6))

for i in range(8):
    plt.subplot(2, 4, i + 1)
    
    # Un-normalize image tensor for display: img * std + mean
    img_tensor = images[i][0] * MNIST_STD[0] + MNIST_MEAN[0]
    img_numpy = img_tensor.numpy()
    
    # Plot image in grayscale
    plt.imshow(img_numpy, cmap='gray')
    plt.title(f"Label: {labels[i].item()}", fontsize=12)
    plt.axis('off')

plt.suptitle("Sample MNIST Training Images (Resized to 224x224)", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Model Architecture Selection and Initialization (ResNet-18)

### What is being done
We instantiate the standard ResNet-18 deep convolutional architecture from `torchvision.models` without using pre-trained weights (`weights=None`).

### Why it is necessary
ResNet-18 is a widely recognized residual CNN architecture equipped with residual skip connections that prevent vanishing gradients. Training from scratch on MNIST provides an un-biased baseline for quantization benchmarking.

### How it contributes to the research
Serves as the baseline deep feature extractor for our model compression and robustness study.

In [ ]:
# Instantiate ResNet-18 without pre-trained weights
model = models.resnet18(weights=None)

print("Standard ResNet-18 architecture initialized without pretrained weights.")

## 11. Adapting Convolutional Input Layer for Single-Channel Grayscale Images

### What is being done
We replace the first convolutional layer (`model.conv1`) of ResNet-18:
- Original: `in_channels = 3` (RGB color images)
- Modified: `in_channels = 1` (Single-channel grayscale MNIST images)

### Why it is necessary
Standard ResNet-18 expects 3-channel RGB images. MNIST digits are single-channel grayscale images. Changing `in_channels` to 1 allows the network to process single-channel inputs directly.

### How it contributes to the research
Customizes standard vision backbones for single-channel edge sensor data.

In [ ]:
# Replace first conv layer for 1-channel grayscale input
model.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

print("Modified model.conv1 to accept 1-channel grayscale input.")

## 12. Adapting Classification Head for 10 Target Classes

### What is being done
We replace the final fully connected linear classification head (`model.fc`):
- Original: `1000` output neurons (ImageNet classes)
- Modified: `10` output neurons (MNIST digits 0 through 9)

### Why it is necessary
The classifier output dimension must match the specific dataset class count.

### How it contributes to the research
Aligns network predictions directly with the target 10-class task.

In [ ]:
# Extract input features dimension from existing fully connected layer
in_features = model.fc.in_features

# Replace final linear layer for 10 output classes
model.fc = nn.Linear(in_features, 10)

print(f"Modified model.fc: {in_features} input features -> {num_classes} output classes.")

## 13. Transferring Model to Compute Device & Architecture Inspection

### What is being done
We transfer the modified ResNet-18 model to the selected compute device (`cuda` or `cpu`) and display the model architecture.

### Why it is necessary
All tensor inputs and model parameters must reside on the same target hardware device for tensor operations to execute.

### How it contributes to the research
Validates correct layer instantiation and parameter placement prior to optimization.

In [ ]:
# Transfer model to targeted compute device
model = model.to(device)

# Display model structure confirmation
print(f"Model transferred to device: {device}")
print("\n=== RESNET-18 MODEL ARCHITECTURE ===")
print(model)

## 14. Loss Function and Optimizer Setup

### What is being done
We configure:
1. `CrossEntropyLoss`: Loss criterion for multi-class digit classification.
2. `Adam`: Optimizer with learning rate `0.001` for adaptive gradient updates.

### Why it is necessary
Loss functions quantify prediction errors while optimizers update model weights to minimize this error.

### How it contributes to the research
Establishes the optimization objective and parameter updating strategy for training.

In [ ]:
# Instantiate Cross Entropy Loss criterion
criterion = nn.CrossEntropyLoss()

# Define learning rate
LEARNING_RATE = 0.001

# Instantiate Adam optimizer over model parameters
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Loss criterion: CrossEntropyLoss | Optimizer: Adam (lr={LEARNING_RATE})")

## 15. Training and Evaluation Routines Implementation

### What is being done
We construct two modular helper functions:
1. `train_one_epoch()`: Executes one training epoch over mini-batches, updating model weights.
2. `evaluate_model()`: Computes classification accuracy, predictions, and ground truth labels over the evaluation loader.

### Why it is necessary
Modularizing training and evaluation routines prevents code duplication and makes training loops transparent.

### How it contributes to the research
Establishes standardized measurement methods for training and testing loops.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Trains the model for one full epoch over the dataloader."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        # Zero parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization step
        loss.backward()
        optimizer.step()
        
        # Accumulate metrics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


def evaluate_model(model, dataloader, device):
    """Evaluates the model and returns accuracy, predictions, and ground truth labels."""
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    accuracy = 100.0 * correct / total
    return accuracy, np.array(all_preds), np.array(all_targets)

print("Training and Evaluation helper functions successfully defined.")

## 16. Training FP32 ResNet-18 Model (3 Epochs)

### What is being done
We train the FP32 ResNet-18 model for `3` epochs as a fast pilot experiment. After each epoch, we record and print:
- Epoch Number
- Training Loss
- Training Accuracy (%)
- Testing Accuracy (%)

### Why it is necessary
Training for 3 epochs produces a baseline weight state for our comparative quantization and robustness study without unnecessary compute expenditure.

### How it contributes to the research
Generates the baseline FP32 model weights for quantization and FGSM attack comparisons.

In [ ]:
# Define number of training epochs
NUM_EPOCHS = 3

print("=== STARTING FP32 RESNET-18 MODEL TRAINING ===")

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    
    # Train for one epoch
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Evaluate on test set
    test_acc, _, _ = evaluate_model(model, test_loader, device)
    
    elapsed_time = time.time() - start_time
    
    # Print metrics after every epoch
    print(f"Epoch [{epoch}/{NUM_EPOCHS}] ({elapsed_time:.2f}s) | "
          f"Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Test Acc: {test_acc:.2f}%")

print("=== FP32 MODEL TRAINING COMPLETED ===")

## 17. FP32 Model Clean Accuracy Evaluation

### What is being done
We evaluate the fully trained FP32 model on the complete MNIST testing dataset (`10,000` samples) to determine final clean accuracy.

### Why it is necessary
Establishing exact unquantized accuracy provides the reference baseline against which quantized INT8 performance will be compared.

### How it contributes to the research
Represents the baseline accuracy ceiling for FP32 precision.

In [ ]:
# Evaluate clean test accuracy of FP32 model
fp32_clean_acc, fp32_preds, fp32_targets = evaluate_model(model, test_loader, device)

print(f"Final Clean Accuracy (FP32 Model): {fp32_clean_acc:.2f}%")

## 18. Model Size and Inference Speed Benchmarking Helpers

### What is being done
We implement two helper benchmarking functions:
1. `get_model_size_mb(model)`: Saves model weights to a temporary file via `torch.save()`, measures size in Megabytes (MB), and deletes the temp file.
2. `measure_inference_speed(model, dataloader, device)`: Times inference using high-precision `time.perf_counter()` to calculate average milliseconds per image.

### Why it is necessary
Accurately measuring file footprint on disk and hardware execution latency is vital for edge deployment feasibility analysis.

### How it contributes to the research
Provides standardized methods for quantifying computational efficiency.

In [ ]:
def get_model_size_mb(model):
    """Measures serialized PyTorch model size in Megabytes (MB)."""
    with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
        torch.save(model.state_dict(), tmp_file.name)
        size_bytes = os.path.getsize(tmp_file.name)
        tmp_path = tmp_file.name
        
    os.remove(tmp_path)
    size_mb = size_bytes / (1024.0 * 1024.0)
    return size_mb


def measure_inference_speed(model, dataloader, device, num_batches=50):
    """Measures average inference time per image in milliseconds (ms)."""
    model.eval()
    total_time = 0.0
    total_images = 0
    
    with torch.no_grad():
        for i, (images, _) in enumerate(dataloader):
            if i >= num_batches:
                break
                
            images = images.to(device)
            batch_size = images.size(0)
            
            start_time = time.perf_counter()
            _ = model(images)
            end_time = time.perf_counter()
            
            total_time += (end_time - start_time)
            total_images += batch_size
            
    avg_ms_per_image = (total_time / total_images) * 1000.0
    return avg_ms_per_image

print("Model size and inference speed measurement functions defined successfully.")

## 19. Benchmarking FP32 Baseline Model Size & Latency

### What is being done
We call our measurement helpers to record the baseline model size (MB) and inference latency (ms per image) for the unquantized FP32 model.

### Why it is necessary
Establishes the quantitative uncompressed reference metrics.

### How it contributes to the research
Fills the baseline row of our efficiency trade-off matrix.

In [ ]:
# Measure FP32 Model Size
fp32_model_size_mb = get_model_size_mb(model)

# Measure FP32 Inference Speed
fp32_inference_time_ms = measure_inference_speed(model, test_loader, device)

print(f"FP32 Model Size:       {fp32_model_size_mb:.2f} MB")
print(f"FP32 Inference Latency: {fp32_inference_time_ms:.4f} ms per image")

## 20. Post-Training Quantization (PTQ) Wrapper Setup

### What is being done
We construct a quantization wrapper module `QuantizableResNet` inserting `QuantStub()` at the input and `DeQuantStub()` at the output, and place the model in evaluation mode on CPU.

### Why it is necessary
Standard Post-Training Static Quantization in PyTorch requires explicit entry/exit stubs to convert floating-point tensors into 8-bit quantized integer tensors during forward execution on CPU.

### How it contributes to the research
Prepares the ResNet-18 model for static INT8 post-training quantization.

In [ ]:
class QuantizableResNet(nn.Module):
    """Wrapper module adding QuantStub and DeQuantStub for static INT8 quantization."""
    def __init__(self, base_model):
        super(QuantizableResNet, self).__init__()
        self.quant = quantization.QuantStub()
        self.base_model = base_model
        self.dequant = quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.base_model(x)
        x = self.dequant(x)
        return x

# Instantiate quantization-wrapped model copy on CPU
model_to_quantize = QuantizableResNet(copy.deepcopy(model)).to('cpu')
model_to_quantize.eval()

print("QuantizableResNet wrapper initialized and placed in evaluation mode on CPU.")

## 21. Attaching Quantization Observers & Calibration

### What is being done
We configure the `fbgemm` CPU quantization backend, attach layer observers via `quantization.prepare()`, and calibrate observer statistics by passing 10 mini-batches of training images.

### Why it is necessary
Calibration allows activation observers to record activation distributions across sample inputs without updating model weights.

### How it contributes to the research
Ensures activation quantization parameters (scale & zero-point) are computed accurately.

In [ ]:
# Set quantization configuration for CPU backend (fbgemm)
model_to_quantize.qconfig = quantization.get_default_qconfig('fbgemm')

# Prepare model for static quantization by inserting observer modules
prepared_model = quantization.prepare(model_to_quantize, inplace=False)
prepared_model.eval()

print("Quantization observers attached. Calibrating over 10 training batches...")

# Run calibration loop
with torch.no_grad():
    for i, (images, _) in enumerate(train_loader):
        if i >= 10:
            break
        prepared_model(images.to('cpu'))

print("Quantization calibration successfully completed!")

## 22. Converting to INT8 and Benchmarking Quantized Performance

### What is being done
We convert the prepared model to a quantized INT8 model using `quantization.convert()` and evaluate:
- INT8 Clean Accuracy (%)
- INT8 Model Size (MB)
- INT8 Inference Latency (ms per image)
- Model Compression Ratio ($	ext{Size}_{	ext{FP32}} / 	ext{Size}_{	ext{INT8}}$)

### Why it is necessary
Evaluates the real-world performance trade-offs resulting from 8-bit integer quantization.

### How it contributes to the research
Quantifies the efficiency vs. accuracy trade-off of PTQ.

In [ ]:
# Convert prepared model to quantized INT8 model
quantized_model = quantization.convert(prepared_model, inplace=False)
print("Model successfully converted to INT8 Quantized Format!")

# Evaluate INT8 clean accuracy on CPU
int8_clean_acc, _, _ = evaluate_model(quantized_model, test_loader, device=torch.device('cpu'))

# Measure INT8 Model Size
int8_model_size_mb = get_model_size_mb(quantized_model)

# Measure INT8 Inference Latency
int8_inference_time_ms = measure_inference_speed(quantized_model, test_loader, device=torch.device('cpu'))

# Calculate Model Compression Ratio
compression_ratio = fp32_model_size_mb / int8_model_size_mb

print("=== INT8 QUANTIZED MODEL METRICS ===")
print(f"INT8 Clean Accuracy:     {int8_clean_acc:.2f}%")
print(f"INT8 Model Size:         {int8_model_size_mb:.2f} MB")
print(f"INT8 Inference Latency:   {int8_inference_time_ms:.4f} ms/image")
print(f"Compression Ratio:       {compression_ratio:.2f}x")

## 23. Implementing Manual FGSM Adversarial Attack Algorithm

### What is being done
We implement the Fast Gradient Sign Method (FGSM) adversarial attack manually ($\epsilon = 0.20$) and create an adversarial evaluation routine:
$$x_{adv} = x + \epsilon \cdot 	ext{sign}(
abla_x L(	heta, x, y))$$

For the INT8 model, adversarial examples are generated using the FP32 surrogate model gradients (standard transfer attack research protocol).

### Why it is necessary
Measures the degradation in accuracy caused by adversarial perturbations.

### How it contributes to the research
Evaluates adversarial robustness trade-offs under quantization.

In [ ]:
def fgsm_attack(image, epsilon, data_grad):
    """Generates Fast Gradient Sign Method (FGSM) adversarial images."""
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    return perturbed_image


def evaluate_fgsm(surrogate_model, eval_model, dataloader, epsilon, surrogate_device, eval_device, max_batches=30):
    """Evaluates model accuracy under FGSM adversarial attack using surrogate gradients."""
    surrogate_model.eval()
    eval_model.eval()
    
    correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()
    
    for i, (images, labels) in enumerate(dataloader):
        if i >= max_batches:
            break
            
        images_surr = images.to(surrogate_device)
        labels_surr = labels.to(surrogate_device)
        images_surr.requires_grad = True
        
        outputs_surr = surrogate_model(images_surr)
        loss = criterion(outputs_surr, labels_surr)
        
        surrogate_model.zero_grad()
        loss.backward()
        
        data_grad = images_surr.grad.data
        perturbed_images = fgsm_attack(images_surr, epsilon, data_grad)
        
        with torch.no_grad():
            perturbed_eval = perturbed_images.to(eval_device)
            labels_eval = labels.to(eval_device)
            
            outputs_eval = eval_model(perturbed_eval)
            _, predicted = torch.max(outputs_eval.data, 1)
            
            total += labels_eval.size(0)
            correct += (predicted == labels_eval).sum().item()
            
    accuracy = 100.0 * correct / total
    return accuracy

print("Manual FGSM attack and evaluation helper functions defined.")

## 24. Benchmarking Adversarial Robustness (FP32 vs INT8 under FGSM $\epsilon=0.20$)

### What is being done
We evaluate FGSM adversarial accuracy ($\epsilon = 0.20$) for both FP32 and INT8 models.

### Why it is necessary
Determines whether INT8 quantization degrades or enhances robustness against adversarial noise.

### How it contributes to the research
Provides empirical security metrics for edge deployment research.

In [ ]:
EPSILON = 0.20

print(f"Evaluating adversarial robustness under FGSM attack (epsilon = {EPSILON})...")

# Evaluate FP32 model adversarial accuracy
fp32_fgsm_acc = evaluate_fgsm(
    surrogate_model=model,
    eval_model=model,
    dataloader=test_loader,
    epsilon=EPSILON,
    surrogate_device=device,
    eval_device=device
)

# Evaluate INT8 model adversarial accuracy
int8_fgsm_acc = evaluate_fgsm(
    surrogate_model=model,
    eval_model=quantized_model,
    dataloader=test_loader,
    epsilon=EPSILON,
    surrogate_device=device,
    eval_device=torch.device('cpu')
)

print(f"FP32 Model FGSM Accuracy (eps={EPSILON}): {fp32_fgsm_acc:.2f}%")
print(f"INT8 Model FGSM Accuracy (eps={EPSILON}): {int8_fgsm_acc:.2f}%")

## 25. Results Aggregation & Export (`results/results.csv`)

### What is being done
We aggregate all experimental metrics into a structured Pandas DataFrame containing 7 columns and 2 rows, and export it to `results/results.csv`.

### Why it is necessary
Tabulating metrics side-by-side facilitates direct comparative analysis and manuscript preparation.

### How it contributes to the research
Stores baseline experimental data systematically for publication.

In [ ]:
# Construct results dictionary
results_data = {
    "Model": ["ResNet18 FP32", "ResNet18 INT8"],
    "Precision": ["FP32", "INT8"],
    "Clean Accuracy": [f"{fp32_clean_acc:.2f}%", f"{int8_clean_acc:.2f}%"],
    "FGSM Accuracy": [f"{fp32_fgsm_acc:.2f}%", f"{int8_fgsm_acc:.2f}%"],
    "Model Size (MB)": [round(fp32_model_size_mb, 2), round(int8_model_size_mb, 2)],
    "Inference Time (ms)": [round(fp32_inference_time_ms, 4), round(int8_inference_time_ms, 4)],
    "Compression Ratio": [1.00, round(compression_ratio, 2)]
}

# Create DataFrame
results_df = pd.DataFrame(results_data)

# Display DataFrame
print("=== EXPERIMENTAL RESULTS COMPARISON TABLE ===")
print(results_df.to_string(index=False))

# Ensure output directory exists and write CSV
results_dir = "./results/"
os.makedirs(results_dir, exist_ok=True)
csv_path = os.path.join(results_dir, "results.csv")
results_df.to_csv(csv_path, index=False)

print(f"Results table successfully saved to: {csv_path}")

## 26. Experimental Metrics Visualizations (Graphs)

### What is being done
We generate and save 5 separate publication-quality bar charts into `./results/`:
1. `clean_accuracy_comparison.png`
2. `fgsm_accuracy_comparison.png`
3. `model_size_comparison.png`
4. `inference_time_comparison.png`
5. `compression_ratio_comparison.png`

Each figure includes explicit titles, axis labels, grids, and numeric value annotations.

### Why it is necessary
Visualizing metrics enables rapid interpretation of trade-offs.

### How it contributes to the research
Produces publication figures for research papers.

In [ ]:
models_labels = ["ResNet18 FP32", "ResNet18 INT8"]

# 1. Clean Accuracy Graph
plt.figure(figsize=(6, 4.5))
bars = plt.bar(models_labels, [fp32_clean_acc, int8_clean_acc], color=['#1f77b4', '#2ca02c'], width=0.45)
plt.title("Clean Accuracy Comparison (FP32 vs INT8)", fontsize=13, fontweight='bold')
plt.ylabel("Clean Accuracy (%)", fontsize=11)
plt.ylim(0, 110)
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "clean_accuracy_comparison.png"), dpi=300)
plt.show()

# 2. FGSM Accuracy Graph
plt.figure(figsize=(6, 4.5))
bars = plt.bar(models_labels, [fp32_fgsm_acc, int8_fgsm_acc], color=['#d62728', '#ff7f0e'], width=0.45)
plt.title("FGSM Adversarial Accuracy Comparison (eps=0.20)", fontsize=13, fontweight='bold')
plt.ylabel("FGSM Accuracy (%)", fontsize=11)
plt.ylim(0, 110)
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "fgsm_accuracy_comparison.png"), dpi=300)
plt.show()

# 3. Model Size Graph
plt.figure(figsize=(6, 4.5))
sizes_mb = [fp32_model_size_mb, int8_model_size_mb]
bars = plt.bar(models_labels, sizes_mb, color=['#9467bd', '#8c564b'], width=0.45)
plt.title("Model Storage Size Comparison (MB)", fontsize=13, fontweight='bold')
plt.ylabel("Model Size (MB)", fontsize=11)
plt.ylim(0, max(sizes_mb) * 1.25)
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + (max(sizes_mb)*0.02), f"{yval:.2f} MB", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "model_size_comparison.png"), dpi=300)
plt.show()

# 4. Inference Time Graph
plt.figure(figsize=(6, 4.5))
latencies_ms = [fp32_inference_time_ms, int8_inference_time_ms]
bars = plt.bar(models_labels, latencies_ms, color=['#e377c2', '#7f7f7f'], width=0.45)
plt.title("Inference Latency Comparison (ms / image)", fontsize=13, fontweight='bold')
plt.ylabel("Latency (ms)", fontsize=11)
plt.ylim(0, max(latencies_ms) * 1.25)
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + (max(latencies_ms)*0.02), f"{yval:.4f} ms", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "inference_time_comparison.png"), dpi=300)
plt.show()

# 5. Compression Ratio Graph
plt.figure(figsize=(6, 4.5))
comp_ratios = [1.0, compression_ratio]
bars = plt.bar(models_labels, comp_ratios, color=['#bcbd22', '#17becf'], width=0.45)
plt.title("Model Compression Ratio (Relative to FP32)", fontsize=13, fontweight='bold')
plt.ylabel("Compression Ratio (x)", fontsize=11)
plt.ylim(0, max(comp_ratios) * 1.25)
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + (max(comp_ratios)*0.02), f"{yval:.2f}x", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "compression_ratio_comparison.png"), dpi=300)
plt.show()

print("All 5 comparison graphs successfully generated and saved to ./results/")

## 27. Final Research Summary & Academic Reproducibility Metadata

### What is being done
We output an objective summary of empirical findings (Clean Accuracy, INT8 Accuracy, Accuracy Drop, FGSM Robustness, Size Reduction, Compression Ratio, Speedup) and log environment specifications (Python version, PyTorch version, TorchVision version, device, random seed, timestamp).

### Why it is necessary
Concise metric reporting and exact environment metadata ensure full transparency and reproducibility for peer review.

### How it contributes to the research
Completes the baseline empirical study notebook.

In [ ]:
import sys

accuracy_drop = fp32_clean_acc - int8_clean_acc
size_reduction_pct = ((fp32_model_size_mb - int8_model_size_mb) / fp32_model_size_mb) * 100.0
speedup_factor = fp32_inference_time_ms / int8_inference_time_ms if int8_inference_time_ms > 0 else 1.0

print("==============================================================")
print("             PILOT RESEARCH EXPERIMENT SUMMARY               ")
print("==============================================================")
print(f"1. FP32 Clean Accuracy:        {fp32_clean_acc:.2f}%")
print(f"2. INT8 Clean Accuracy:        {int8_clean_acc:.2f}%")
print(f"3. Accuracy Drop:              {accuracy_drop:.2f}%")
print(f"4. FP32 FGSM Robustness:       {fp32_fgsm_acc:.2f}%")
print(f"5. INT8 FGSM Robustness:       {int8_fgsm_acc:.2f}%")
print(f"6. FP32 Model Size:            {fp32_model_size_mb:.2f} MB")
print(f"7. INT8 Model Size:            {int8_model_size_mb:.2f} MB")
print(f"8. Storage Size Reduction:     {size_reduction_pct:.2f}%")
print(f"9. Compression Ratio:          {compression_ratio:.2f}x")
print(f"10. FP32 Inference Speed:      {fp32_inference_time_ms:.4f} ms/image")
print(f"11. INT8 Inference Speed:      {int8_inference_time_ms:.4f} ms/image")
print(f"12. Inference Speedup Factor:  {speedup_factor:.2f}x")
print("==============================================================")

print("\n=== ACADEMIC REPRODUCIBILITY METADATA ===")
print(f"Python Version:     {sys.version.split()[0]}")
print(f"PyTorch Version:    {torch.__version__}")
print(f"TorchVision Version:{transforms.__file__}")
print(f"Compute Device:     {device}")
print(f"Random Seed:        {SEED}")
print(f"Timestamp:          {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=========================================")